In [ ]:
from math import *
from build123d import *
from cad_utils import *
from pathlib import Path

set_port(3939)
set_viewer_config(port=3939)

In [ ]:
PCB_NAME = "esp-butt"
REPO_ROOT = Path(".").resolve().parent
DOCS_PUBLIC = REPO_ROOT / "docs" / "public"
MODELS_DIR = DOCS_PUBLIC / "models"
PCB_DIR = REPO_ROOT / "pcb"

from io import BytesIO
from OCP.XCAFDoc import XCAFDoc_DocumentTool

def dump(doc):
  shape_tool = XCAFDoc_DocumentTool.ShapeTool_s(doc.Main())
  b = BytesIO()
  shape_tool.Dump(b, True)
  print(b.getvalue().decode())

In [ ]:
PCB = load_pcb(
  "esp-butt",
  "../pcb/esp-butt",
  "../pcb/export",
  {"AZ-Delivery_-_OLED13_IIC": "J1 Screen"},
)

pcb_slider_levers = [PCB.find("R7", "Lever"), PCB.find("R8", "Lever")]
pcb_encoder = PCB.find("S1")
pcb_switch  = PCB.find("S2")

encoder_knob = import_step(MODELS_DIR / "encoder_knob.step")
case_top = import_step(MODELS_DIR / "case_top.step")
case_bottom = import_step(MODELS_DIR / "case_bottom.step")
slider_knob = import_step(MODELS_DIR / "slider_knob.step")
power_switch_cap = import_step(MODELS_DIR / "power_switch_cap.step")

In [ ]:
reset_show()

slider_knob1 = Compound(
  label="slider_knob1", children=[fast_copy(c) for c in slider_knob.children]
)
slider_knob1.location = Location(pcb_slider_levers[0].center() + Vector(0, -45, 4.5), (0, 0, 90))
slider_knob2 = Compound(
  label="slider_knob2", children=[fast_copy(c) for c in slider_knob.children]
)
slider_knob2.location = Location(pcb_slider_levers[1].center() + Vector(0, -45, 4.5), (0, 0, 90))

assembly = Compound(None, label="printed_parts", children=[
  fast_copy(case_top),
  fast_copy(case_bottom),
  slider_knob1,
  slider_knob2,
  copy_located(encoder_knob, Location(pcb_encoder.center() + Vector(0, 0, 8))),
  copy_located(power_switch_cap, Location(pcb_switch.faces().group_by(Axis.Y)[-1].first.center() + Vector(0, -3, 0), (270, 0, 0))),
])

show_object(assembly)

export_step(assembly, MODELS_DIR / "printed_parts.step", write_pcurves=False)
step_doc = import_step_doc(MODELS_DIR / "printed_parts.step")
export_gltf_doc(step_doc, MODELS_DIR / "printed_parts.glb", scale=100)